# Hospital Demand Forecasting — Pune Community Clinic Network

**Synthetic demonstration project.** No official internship dataset was supplied, so this project creates fictional encounter records. Do not represent them as real patient data or an official hospital forecast.

**Scope:** 4 fictional hospitals, 3 service lines (OPD, Emergency, Inpatient), approximately 500,000 encounter rows, weekly aggregation, 13-week forecast.

## Workflow
1. Review data quality and clean duplicates/missing values.
2. Aggregate weekly demand by hospital and service line.
3. Explore weekly demand trend.
4. Compare Seasonal Naive, ETS and Random Forest lag-feature models on the final 13 weeks.
5. Produce segment forecasts with approximate 80%/95% residual ranges.
6. Translate volume into illustrative staffing, bed and follow-up planning metrics.

Run from the project root. Install: `pip install pandas numpy matplotlib scikit-learn statsmodels jupyter`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
ROOT=Path(".")
raw=pd.read_csv(ROOT/"data/synthetic_hospital_encounters_500k.csv",parse_dates=["visit_date"])
print(raw.shape); display(raw.head())

## 1. Data quality and cleaning

The source deliberately contains duplicate visit IDs, missing dates and missing ages. Duplicate IDs are removed; records missing date, hospital or service line are excluded; missing age is median-imputed. Age is not used as the forecast target.

In [ ]:
display(pd.DataFrame({"missing":raw.isna().sum(),"missing_pct":(raw.isna().mean()*100).round(2)}))
print("Duplicate IDs:",raw.visit_id.duplicated().sum())
clean=raw.drop_duplicates("visit_id").dropna(subset=["visit_date","hospital","service_line"]).copy()
clean["age"]=clean.age.fillna(clean.age.median())
clean["week_start"]=clean.visit_date.dt.to_period("W-SUN").apply(lambda x:x.start_time)
print("Clean rows:",len(clean))

## 2. Weekly aggregation and complete panel

In [ ]:
weekly=clean.groupby(["week_start","hospital","service_line"]).agg(
 visits=("visit_id","count"),admissions=("admission_flag","sum"),lab_tests=("lab_tests","sum"),
 billing_inr=("billing_amount_inr","sum"),followups=("followup_required","sum")).reset_index()
weeks=pd.date_range(weekly.week_start.min(),weekly.week_start.max(),freq="W-MON")
idx=pd.MultiIndex.from_product([weeks,sorted(clean.hospital.unique()),sorted(clean.service_line.unique())],
 names=["week_start","hospital","service_line"])
weekly=weekly.set_index(["week_start","hospital","service_line"]).reindex(idx,fill_value=0).reset_index()
weekly.to_csv(ROOT/"outputs/weekly_hospital_demand.csv",index=False)
display(weekly.head())

## 3. Exploratory analysis

The synthetic dataset has no verified local holiday calendar. Holiday effects (Diwali, Holi, etc.) must be added from validated dates before drawing conclusions about festival-driven demand.

In [ ]:
weekly.groupby("week_start").visits.sum().plot(figsize=(10,4),title="Weekly visits across four hospitals")
plt.ylabel("Visits"); plt.tight_layout(); plt.show()
display(weekly.groupby(["hospital","service_line"]).visits.agg(["mean","std","min","max"]).round(1))

## 4. Model comparison

The final 13 weeks are held out chronologically. This starter version uses one holdout origin, not multiple rolling origins. MAPE excludes zero-actual weeks. Bias = forecast − actual; positive bias means over-forecasting.

In [ ]:
def mape(a,p):
 a=np.asarray(a,float); p=np.asarray(p,float); m=a!=0
 return np.mean(np.abs((a[m]-p[m])/a[m]))*100 if m.any() else np.nan
def ets(y,h):
 try:return np.maximum(0,ExponentialSmoothing(y,trend="add",seasonal="add",seasonal_periods=52,initialization_method="estimated").fit().forecast(h))
 except:return np.maximum(0,ExponentialSmoothing(y,trend="add",initialization_method="estimated").fit().forecast(h))
def rf_forecast(hist,h):
 hist=list(np.asarray(hist,float))
 if len(hist)<53:return np.repeat(hist[-1],h)
 X=[]; Y=[]
 for t in range(52,len(hist)):
  X.append([hist[t-1],hist[t-2],hist[t-4],hist[t-13],hist[t-26],hist[t-52],np.mean(hist[t-4:t]),np.mean(hist[t-13:t])]); Y.append(hist[t])
 model=RandomForestRegressor(n_estimators=30,max_depth=5,min_samples_leaf=3,random_state=42).fit(X,Y)
 out=[]
 for _ in range(h):
  x=[hist[-1],hist[-2],hist[-4],hist[-13],hist[-26],hist[-52],np.mean(hist[-4:]),np.mean(hist[-13:])]
  v=max(0,float(model.predict([x])[0])); out.append(v); hist.append(v)
 return np.array(out)
metrics=[]; forecasts=[]
future=pd.date_range(weekly.week_start.max()+pd.Timedelta(days=7),periods=13,freq="W-MON")
for (hospital,service),g in weekly.groupby(["hospital","service_line"]):
 y=g.sort_values("week_start").visits.to_numpy(float); train,test=y[:-13],y[-13:]
 sn=np.array([train[-52+i] if len(train)>=52 else train[-1] for i in range(13)])
 preds={"Seasonal Naive":sn,"ETS":ets(train,13),"Random Forest":rf_forecast(train,13)}
 for model,p in preds.items(): metrics.append({"hospital":hospital,"service_line":service,"model":model,"mape_pct":mape(test,p),"bias_visits":np.mean(p-test)})
 chosen=min(preds,key=lambda k:mape(test,preds[k]) if not np.isnan(mape(test,preds[k])) else np.inf)
 full={"Seasonal Naive":np.array([y[-52+i] if len(y)>=52 else y[-1] for i in range(13)]),"ETS":ets(y,13),"Random Forest":rf_forecast(y,13)}[chosen]
 sd=np.std(test-preds[chosen],ddof=1)
 for wk,v in zip(future,full): forecasts.append({"week_start":wk,"hospital":hospital,"service_line":service,"forecast_visits":v,
 "lower_80":max(0,v-1.282*sd),"upper_80":v+1.282*sd,"lower_95":max(0,v-1.96*sd),"upper_95":v+1.96*sd,"selected_model":chosen})
backtest=pd.DataFrame(metrics); forecast=pd.DataFrame(forecasts)
backtest.to_csv(ROOT/"outputs/model_backtest_results.csv",index=False); forecast.to_csv(ROOT/"outputs/13_week_forecast.csv",index=False)
display(backtest.groupby("model")[["mape_pct","bias_visits"]].mean().round(2))

## 5. Forecast visualisation and planning table

Intervals use the standard deviation of the 13 holdout residuals with normal multipliers (1.282 and 1.96). These are approximate, not calibrated guarantees. Planning conversions below are illustrative assumptions only and are not clinical staffing standards.

In [ ]:
backtest.groupby("model").mape_pct.mean().sort_values().plot(kind="bar",title="Mean holdout MAPE",ylabel="MAPE (%)"); plt.tight_layout(); plt.show()
forecast.groupby("week_start").forecast_visits.sum().plot(marker="o",title="13-week network forecast",ylabel="Visits"); plt.tight_layout(); plt.show()
plan=forecast.pivot_table(index=["week_start","hospital"],columns="service_line",values="forecast_visits").reset_index()
for c in ["OPD","Emergency","Inpatient"]:
 if c not in plan: plan[c]=0
plan["recommended_opd_staff_slots"]=np.ceil(plan.OPD/25).astype(int)
plan["recommended_emergency_staff_slots"]=np.ceil(plan.Emergency/35).astype(int)
plan["estimated_inpatient_beds"]=np.ceil(plan.Inpatient*2.2/7).astype(int)
plan["followup_call_capacity"]=np.ceil((plan.OPD*.35+plan.Inpatient*.68)*.9).astype(int)
plan.to_csv(ROOT/"outputs/weekly_planning_table.csv",index=False)
display(plan.head())

## 6. Limitations and next steps

1. All records are synthetic; they do not describe real Pune hospitals.
2. Add multiple expanding-window rolling origins (4–6 recommended) before treating the model comparison as robust.
3. Add a verified local holiday/event calendar to evaluate Diwali, Holi and other effects.
4. Diagnosis text is simulated; real diagnoses require standardisation and privacy controls.
5. Residual-based intervals need calibration checks; consider bootstrap or conformal intervals.
6. Staffing, bed and call conversion factors must be replaced with hospital-approved operational standards.

## Deliverables
See `data/`, `outputs/`, and the accompanying Word report.